# Day 033 | Handling Imbalanced Datasets## 100 Days of ML for Geotechnical Engineering**Phase:** Data Science Essentials**Author:** Ripon Chandra Malo  **Date:** 2026---## What We Will Learn Today| Concept | What It Means | Geotechnical Example ||---------|--------------|---------------------|| **Class Imbalance** | One class has far fewer samples | 15% liquefaction vs 85% no-liquefaction || **SMOTE** | Generate synthetic minority samples | Create realistic "liquefied" data points || **Undersampling** | Remove majority class samples | Reduce "no-liquefaction" samples to balance || **class_weight** | Penalize misclassifying rare events more | Make model pay attention to failures || **Precision** | Of predicted positives, how many are correct? | Of predicted liquefaction cases, how many actually liquefied? || **Recall** | Of actual positives, how many were found? | Of all actual liquefaction cases, how many did we catch? || **F1 Score** | Harmonic mean of precision and recall | Balanced metric for imbalanced data |---## Our Geotechnical ProblemSoil liquefaction is a rare but catastrophic event during earthquakes. In our dataset of 200 sites, only about 15% experienced liquefaction. If our model simply predicts "no liquefaction" every time, it would be 85% accurate — but completely useless for safety!**Goal:** Learn techniques to handle this class imbalance so our model actually learns to detect the rare but critical liquefaction events.---

## Part 1: Understanding Class ImbalanceIn geotechnical engineering, imbalanced datasets are the norm, not the exception:- **Liquefaction:** ~5-15% of sites liquefy during earthquakes- **Slope failures:** <5% of slopes fail in a given period- **Foundation failures:** Extremely rare (<1%) but devastating- **Retaining wall failures:** <3% of walls experience significant distressThe problem: standard ML algorithms optimize for overall accuracy, which means they learn to predict the majority class and ignore the minority class entirely.---

In [ ]:
# ─────────────────────────────────────────────────────# LOAD AND EXAMINE THE LIQUEFACTION DATASET# ─────────────────────────────────────────────────────import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitimport warningswarnings.filterwarnings('ignore')df = pd.read_csv("liquefaction_data.csv")print("Liquefaction Dataset")print("=" * 55)print(f"Total samples: {len(df)}")print(f"Features: {list(df.columns)}")print()print(df.head(10))

In [ ]:
# ─────────────────────────────────────────────────────# CLASS DISTRIBUTION — The imbalance# ─────────────────────────────────────────────────────class_counts = df['liquefied'].value_counts()class_pct = df['liquefied'].value_counts(normalize=True) * 100print("Class Distribution:")print("─" * 40)print(f"  Not Liquefied (0): {class_counts[0]:4d} samples ({class_pct[0]:.1f}%)")print(f"  Liquefied (1)    : {class_counts[1]:4d} samples ({class_pct[1]:.1f}%)")print(f"  Imbalance ratio  : {class_counts[0] / class_counts[1]:.1f} : 1")print()# ── Visualize ──fig, axes = plt.subplots(1, 2, figsize=(12, 4))# Bar chartcolors = ['#4CAF50', '#F44336']bars = axes[0].bar(['Not Liquefied\n(Class 0)', 'Liquefied\n(Class 1)'],                      class_counts.values, color=colors, edgecolor='black', alpha=0.8)axes[0].set_ylabel('Number of Samples')axes[0].set_title('Class Distribution (Count)', fontweight='bold')for bar, count in zip(bars, class_counts.values):    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,                   str(count), ha='center', fontweight='bold')# Pie chartaxes[1].pie(class_counts.values, labels=['Not Liquefied', 'Liquefied'],             colors=colors, autopct='%1.1f%%', startangle=90,             explode=(0, 0.1), shadow=True, textprops={'fontsize': 11})axes[1].set_title('Class Distribution (Percentage)', fontweight='bold')plt.tight_layout()plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')plt.show()print("This severe imbalance means a 'predict all 0' model")print(f"would achieve {class_pct[0]:.1f}% accuracy — misleadingly high!")

## Part 2: The Accuracy TrapLet us prove why accuracy is a terrible metric for imbalanced data.---

In [ ]:
# ─────────────────────────────────────────────────────# THE ACCURACY TRAP — Dummy Classifier# ─────────────────────────────────────────────────────from sklearn.dummy import DummyClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix# ── Prepare data ──feature_cols = ['depth', 'N_value', 'fines_content', 'water_table',                 'acceleration', 'magnitude']X = df[feature_cols]y = df['liquefied']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)# ── A model that ALWAYS predicts "not liquefied" ──dummy = DummyClassifier(strategy='most_frequent')dummy.fit(X_train, y_train)y_pred_dummy = dummy.predict(X_test)print("Dummy Model (always predicts 'Not Liquefied')")print("=" * 55)print(f"Accuracy: {accuracy_score(y_test, y_pred_dummy):.1%}")print()print("Confusion Matrix:")cm = confusion_matrix(y_test, y_pred_dummy)print(f"  Predicted →     Not Liq.   Liquefied")print(f"  Actual Not Liq.   {cm[0,0]:4d}       {cm[0,1]:4d}")print(f"  Actual Liquefied  {cm[1,0]:4d}       {cm[1,1]:4d}")print()print("Classification Report:")print(classification_report(y_test, y_pred_dummy,                              target_names=['Not Liquefied', 'Liquefied']))print("PROBLEM: High accuracy but ZERO recall for liquefaction!")print("It would miss every single dangerous site!")

## Part 3: Better Metrics for Imbalanced DataFor imbalanced classification, we need metrics that care about the minority class:| Metric | Formula | What It Tells Us ||--------|---------|-----------------|| **Precision** | TP / (TP + FP) | When we predict liquefaction, how often are we right? || **Recall** | TP / (TP + FN) | Of all actual liquefaction cases, how many did we catch? || **F1 Score** | 2 × (P × R) / (P + R) | Balance between precision and recall || **ROC-AUC** | Area under ROC curve | Overall ranking ability across thresholds |In geotechnical safety, **recall is critical** — missing a liquefaction event (false negative) is far more dangerous than a false alarm (false positive).---

In [ ]:
# ─────────────────────────────────────────────────────# BASELINE MODEL — Logistic Regression (no balancing)# ─────────────────────────────────────────────────────from sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score# ── Build a baseline pipeline ──baseline = Pipeline([    ('scaler', StandardScaler()),    ('model', LogisticRegression(random_state=42, max_iter=1000))])baseline.fit(X_train, y_train)y_pred_base = baseline.predict(X_test)y_prob_base = baseline.predict_proba(X_test)[:, 1]print("Baseline Logistic Regression (no balancing)")print("=" * 55)print(f"  Accuracy  : {accuracy_score(y_test, y_pred_base):.3f}")print(f"  Precision : {precision_score(y_test, y_pred_base, zero_division=0):.3f}")print(f"  Recall    : {recall_score(y_test, y_pred_base):.3f}")print(f"  F1 Score  : {f1_score(y_test, y_pred_base):.3f}")print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_base):.3f}")print()print("Confusion Matrix:")cm = confusion_matrix(y_test, y_pred_base)print(f"  Predicted →     Not Liq.   Liquefied")print(f"  Actual Not Liq.   {cm[0,0]:4d}       {cm[0,1]:4d}")print(f"  Actual Liquefied  {cm[1,0]:4d}       {cm[1,1]:4d}")

## Part 4: SMOTE — Synthetic Minority OversamplingSMOTE (Synthetic Minority Oversampling TEchnique) creates **new synthetic samples** for the minority class by interpolating between existing minority samples and their nearest neighbors.How it works:1. Pick a minority sample2. Find its k nearest minority neighbors3. Create a new sample along the line between themThis gives the model more minority examples to learn from without just duplicating existing ones.---

In [ ]:
# ─────────────────────────────────────────────────────# SMOTE OVERSAMPLING# ─────────────────────────────────────────────────────from imblearn.over_sampling import SMOTEprint("Before SMOTE:")print(f"  Training set: {len(y_train)} samples")print(f"  Class 0: {(y_train == 0).sum()}, Class 1: {(y_train == 1).sum()}")print()# ── Apply SMOTE to training data only ──smote = SMOTE(random_state=42)X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)print("After SMOTE:")print(f"  Training set: {len(y_train_smote)} samples")print(f"  Class 0: {(y_train_smote == 0).sum()}, Class 1: {(y_train_smote == 1).sum()}")print()# ── Train with SMOTE-balanced data ──model_smote = Pipeline([    ('scaler', StandardScaler()),    ('model', LogisticRegression(random_state=42, max_iter=1000))])model_smote.fit(X_train_smote, y_train_smote)y_pred_smote = model_smote.predict(X_test)y_prob_smote = model_smote.predict_proba(X_test)[:, 1]print("SMOTE + Logistic Regression:")print("─" * 40)print(f"  Accuracy  : {accuracy_score(y_test, y_pred_smote):.3f}")print(f"  Precision : {precision_score(y_test, y_pred_smote, zero_division=0):.3f}")print(f"  Recall    : {recall_score(y_test, y_pred_smote):.3f}")print(f"  F1 Score  : {f1_score(y_test, y_pred_smote):.3f}")print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_smote):.3f}")print()cm = confusion_matrix(y_test, y_pred_smote)print(f"  Predicted →     Not Liq.   Liquefied")print(f"  Actual Not Liq.   {cm[0,0]:4d}       {cm[0,1]:4d}")print(f"  Actual Liquefied  {cm[1,0]:4d}       {cm[1,1]:4d}")print()print("SMOTE typically improves recall significantly!")

## Part 5: Random UndersamplingInstead of creating more minority samples, we can **remove majority samples** to balance the classes. This is simpler but loses information.---

In [ ]:
# ─────────────────────────────────────────────────────# RANDOM UNDERSAMPLING# ─────────────────────────────────────────────────────from imblearn.under_sampling import RandomUnderSamplerunder = RandomUnderSampler(random_state=42)X_train_under, y_train_under = under.fit_resample(X_train, y_train)print("After Undersampling:")print(f"  Training set: {len(y_train_under)} samples")print(f"  Class 0: {(y_train_under == 0).sum()}, Class 1: {(y_train_under == 1).sum()}")print(f"  Samples removed: {len(y_train) - len(y_train_under)}")print()# ── Train with undersampled data ──model_under = Pipeline([    ('scaler', StandardScaler()),    ('model', LogisticRegression(random_state=42, max_iter=1000))])model_under.fit(X_train_under, y_train_under)y_pred_under = model_under.predict(X_test)y_prob_under = model_under.predict_proba(X_test)[:, 1]print("Undersampling + Logistic Regression:")print("─" * 40)print(f"  Accuracy  : {accuracy_score(y_test, y_pred_under):.3f}")print(f"  Precision : {precision_score(y_test, y_pred_under, zero_division=0):.3f}")print(f"  Recall    : {recall_score(y_test, y_pred_under):.3f}")print(f"  F1 Score  : {f1_score(y_test, y_pred_under):.3f}")print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_under):.3f}")

## Part 6: class_weight Parameter — The Easiest FixMany sklearn classifiers have a `class_weight` parameter. Setting `class_weight='balanced'` automatically adjusts the loss function to penalize misclassification of the minority class more heavily.**No resampling needed** — the algorithm itself learns to pay more attention to rare events.---

In [ ]:
# ─────────────────────────────────────────────────────# CLASS_WEIGHT='balanced'# ─────────────────────────────────────────────────────model_balanced = Pipeline([    ('scaler', StandardScaler()),    ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])model_balanced.fit(X_train, y_train)y_pred_bal = model_balanced.predict(X_test)y_prob_bal = model_balanced.predict_proba(X_test)[:, 1]print("class_weight='balanced' + Logistic Regression:")print("─" * 40)print(f"  Accuracy  : {accuracy_score(y_test, y_pred_bal):.3f}")print(f"  Precision : {precision_score(y_test, y_pred_bal, zero_division=0):.3f}")print(f"  Recall    : {recall_score(y_test, y_pred_bal):.3f}")print(f"  F1 Score  : {f1_score(y_test, y_pred_bal):.3f}")print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_bal):.3f}")print()# Show the weightsn_0 = (y_train == 0).sum()n_1 = (y_train == 1).sum()w_0 = len(y_train) / (2 * n_0)w_1 = len(y_train) / (2 * n_1)print(f"Computed weights: Class 0 = {w_0:.3f}, Class 1 = {w_1:.3f}")print(f"Minority class (liquefied) gets {w_1/w_0:.1f}x more weight in the loss!")

## Part 7: Comparing All ApproachesLet us compare all four strategies side by side.---

In [ ]:
# ─────────────────────────────────────────────────────# SIDE-BY-SIDE COMPARISON# ─────────────────────────────────────────────────────results = {    'Baseline': {'acc': accuracy_score(y_test, y_pred_base),                  'prec': precision_score(y_test, y_pred_base, zero_division=0),                  'rec': recall_score(y_test, y_pred_base),                  'f1': f1_score(y_test, y_pred_base),                  'auc': roc_auc_score(y_test, y_prob_base)},    'SMOTE': {'acc': accuracy_score(y_test, y_pred_smote),               'prec': precision_score(y_test, y_pred_smote, zero_division=0),               'rec': recall_score(y_test, y_pred_smote),               'f1': f1_score(y_test, y_pred_smote),               'auc': roc_auc_score(y_test, y_prob_smote)},    'Undersampling': {'acc': accuracy_score(y_test, y_pred_under),                       'prec': precision_score(y_test, y_pred_under, zero_division=0),                       'rec': recall_score(y_test, y_pred_under),                       'f1': f1_score(y_test, y_pred_under),                       'auc': roc_auc_score(y_test, y_prob_under)},    'class_weight': {'acc': accuracy_score(y_test, y_pred_bal),                      'prec': precision_score(y_test, y_pred_bal, zero_division=0),                      'rec': recall_score(y_test, y_pred_bal),                      'f1': f1_score(y_test, y_pred_bal),                      'auc': roc_auc_score(y_test, y_prob_bal)},}print("Method Comparison for Liquefaction Prediction")print("=" * 70)print(f"{'Method':16s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'ROC-AUC':>10s}")print("─" * 70)for method, scores in results.items():    print(f"{method:16s} {scores['acc']:10.3f} {scores['prec']:10.3f} {scores['rec']:10.3f} {scores['f1']:10.3f} {scores['auc']:10.3f}")print()print("Key insights:")print("  - Baseline may have highest accuracy but worst recall")print("  - SMOTE and class_weight typically improve recall significantly")print("  - For safety-critical tasks like liquefaction, maximize recall!")

In [ ]:
# ─────────────────────────────────────────────────────# VISUALIZATION OF RESULTS# ─────────────────────────────────────────────────────fig, ax = plt.subplots(figsize=(10, 5))methods = list(results.keys())metrics = ['acc', 'prec', 'rec', 'f1', 'auc']metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']x = np.arange(len(methods))width = 0.15colors = ['#2196F3', '#FF9800', '#F44336', '#4CAF50', '#9C27B0']for i, (metric, label, color) in enumerate(zip(metrics, metric_labels, colors)):    values = [results[m][metric] for m in methods]    ax.bar(x + i * width, values, width, label=label, color=color, alpha=0.8, edgecolor='black')ax.set_xlabel('Method')ax.set_ylabel('Score')ax.set_title('Imbalanced Data Handling: Method Comparison', fontweight='bold')ax.set_xticks(x + 2 * width)ax.set_xticklabels(methods)ax.legend(loc='lower right')ax.set_ylim(0, 1.1)ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)plt.tight_layout()plt.savefig('imbalance_comparison.png', dpi=100, bbox_inches='tight')plt.show()

---## Practice Exercises### Exercise 1: Confusion matrix visualizationUse `sklearn.metrics.ConfusionMatrixDisplay` to plot confusion matrices for all four methods in a 2x2 subplot grid.### Exercise 2: Threshold tuningFor the class_weight model, vary the classification threshold from 0.1 to 0.9 and plot precision vs recall. Find the threshold that gives the best F1 score.### Exercise 3: SMOTE variantsTry `imblearn.over_sampling.BorderlineSMOTE` and `imblearn.over_sampling.ADASYN`. Compare their F1 scores to standard SMOTE.---

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 1: Confusion matrix visualization# ─────────────────────────────────────────────────────# from sklearn.metrics import ConfusionMatrixDisplay# fig, axes = plt.subplots(2, 2, figsize=(10, 8))# for ax, (name, y_pred) in zip(axes.flat,#     [('Baseline', y_pred_base), ('SMOTE', y_pred_smote),#      ('Undersampling', y_pred_under), ('class_weight', y_pred_bal)]):#     ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)#     ax.set_title(name)# plt.tight_layout()# plt.show()

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 2: Threshold tuning# ─────────────────────────────────────────────────────# thresholds = np.arange(0.1, 0.91, 0.05)# f1_scores = []# for t in thresholds:#     y_pred_t = (y_prob_bal >= t).astype(int)#     f1_scores.append(f1_score(y_test, y_pred_t))# best_t = thresholds[np.argmax(f1_scores)]# print(f"Best threshold: {best_t:.2f}, F1: {max(f1_scores):.3f}")

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 3: SMOTE variants# ─────────────────────────────────────────────────────# from imblearn.over_sampling import BorderlineSMOTE, ADASYN# for name, sampler in [('BorderlineSMOTE', BorderlineSMOTE(random_state=42)),#                        ('ADASYN', ADASYN(random_state=42))]:#     X_res, y_res = sampler.fit_resample(X_train, y_train)#     pipe = Pipeline([('scaler', StandardScaler()),#                       ('model', LogisticRegression(random_state=42, max_iter=1000))])#     pipe.fit(X_res, y_res)#     y_pred = pipe.predict(X_test)#     print(f"{name}: F1={f1_score(y_test, y_pred):.3f}")

---## Key Takeaways1. **Class imbalance is the default in geotechnical engineering.** Failures, liquefaction, and collapses are rare events. A model that ignores the minority class is dangerous.2. **Accuracy is misleading** for imbalanced datasets. Use F1 score, precision, recall, and ROC-AUC instead.3. **SMOTE** creates synthetic minority samples by interpolating between neighbors. It works well but must only be applied to training data.4. **class_weight='balanced'** is the simplest fix — no resampling needed. Many sklearn models support it.5. **In geotechnical safety, recall matters most.** Missing a liquefaction event (false negative) is far worse than a false alarm (false positive). Tune your threshold accordingly.---## How This Connects to ML| What We Did Today | What It Becomes in ML ||---|---|| Class distribution analysis | First step in any classification project || SMOTE oversampling | Standard technique for rare event prediction || class_weight parameter | Built-in support in Random Forest, SVM, etc. || Precision vs recall trade-off | Safety-critical decision making || Confusion matrix | Model evaluation and error analysis || Threshold tuning | Deployment decision: sensitivity vs specificity |---*Day 033 of 100 — Keep going!*